<a href="https://colab.research.google.com/github/ChristanSanchez/Special-Oops-App-Development/blob/feature-backend/Group%205%2C%20CPE41S3%2C%20CPE%20028%20(Code).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

MODEL_PATH = "mnist_model.keras"

# EXTRACT

print("Loading MNIST dataset...")
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# TRANSFORM

# Normalize pixel values from 0-255 to 0-1
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Add channel dimension
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(10, activation="softmax")
])

model.compile( optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# LOAD
model.fit( x_train, y_train, epochs=5, batch_size=128, validation_split=0.1)

Loading MNIST dataset...
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 44s 101ms/step - accuracy: 0.9190 - loss: 0.2626 - val_accuracy: 0.9835 - val_loss: 0.0597
Epoch 2/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 46s 110ms/step - accuracy: 0.9758 - loss: 0.0787 - val_accuracy: 0.9885 - val_loss: 0.0390
Epoch 3/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 43s 101ms/step - accuracy: 0.9826 - loss: 0.0565 - val_accuracy: 0.9872 - val_loss: 0.0411
Epoch 4/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 41s 98ms/step - accuracy: 0.9864 - loss: 0.0444 - val_accuracy: 0.9897 - val_loss: 0.0397
Epoch 5/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 82s 99ms/step - accuracy: 0.9883 - loss: 0.0381 - val_accuracy: 0.9898 - val_loss: 0.0362


In [2]:
# Test Accuracy
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
model.save(MODEL_PATH)
print("Model saved!")

Test Accuracy: 98.92%
Model saved!


In [ ]:
# Camera
print("Opening camera...")
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: Could not open camera.")
    exit()
while True:
    ret, frame = cap.read()
    if not ret:
        break
    # Flip camera so it behaves like a mirror
    frame = cv2.flip(frame, 1)
    # Create grayscale image
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # Threshold image
    _, thresh = cv2.threshold(
        gray,
        100,
        255,
        cv2.THRESH_BINARY_INV
    )
    # Find contours
    contours, _ = cv2.findContours(
        thresh,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )
    # Find the largest contour
    digit_contour = None
    if contours:
        digit_contour = max(
            contours,
            key=cv2.contourArea
        )
    prediction = None
    confidence = 0
    if digit_contour is not None:
        area = cv2.contourArea(digit_contour)
        # Ignore very small objects/noise
        if area > 500:
            x, y, w, h = cv2.boundingRect(digit_contour)
            # Draw bounding box
            cv2.rectangle(
                frame,
                (x, y),
                (x + w, y + h),
                (0, 255, 0),
                2
            )
            # Extract digit
            digit = thresh[y:y+h, x:x+w]
            # Add padding
            padding = 20
            digit = cv2.copyMakeBorder(
                digit,
                padding,
                padding,
                padding,
                padding,
                cv2.BORDER_CONSTANT,
                value=0
            )
            # Resize while keeping digit proportions
            digit = cv2.resize(digit, (28, 28), ninterpolation=cv2.INTER_AREA)
            # Normalize
            digit = digit.astype("float32") / 255.0
            # Reshape for CNN
            digit = digit.reshape(1, 28, 28, 1)
            # Predict
            probabilities = model.predict(digit, verbose=0)[0]
            prediction = np.argmax(probabilities)
            confidence = probabilities[prediction] * 100
            # Display prediction
            text = f"Number: {prediction} ({confidence:.1f}%)"
            cv2.putText(frame, text, (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

    # Show camera
    cv2.imshow("MNIST Number Recognition",frame)

    # Press Q to quit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()